MLP with out feature engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # # Create lag features with a window size of 24
    # lags = 24
    # for i in range(1, lags + 1):
    #     df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # # Add rolling statistics (mean, std, skewness) with a window size of 24
    # df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    # df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    # df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


MLP BM + Rolling statistics features only

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # # Create lag features with a window size of 24
    # lags = 24
    # for i in range(1, lags + 1):
    #     df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # Add rolling statistics (mean, std, skewness) with a window size of 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


BM + Time-Based Features Only

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # # Create lag features with a window size of 24
    # lags = 24
    # for i in range(1, lags + 1):
    #     df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # # Add rolling statistics (mean, std, skewness) with a window size of 24
    # df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    # df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    # df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    # Add hourly and daily aggregated features
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['weekday'] = df.index.weekday
    df['month'] = df.index.month

    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


BM + Lagging Features Only

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # Create lag features with a window size of 24
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # # Add rolling statistics (mean, std, skewness) with a window size of 24
    # df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    # df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    # df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


BM+ Lagging Features + Rolling Statistics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # Create lag features with a window size of 24
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # Add rolling statistics (mean, std, skewness) with a window size of 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


BM + Lagging Features + Time-Based Features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # Create lag features with a window size of 24
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # # Add rolling statistics (mean, std, skewness) with a window size of 24
    # df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    # df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    # df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    # Add hourly and daily aggregated features
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['weekday'] = df.index.weekday
    df['month'] = df.index.month

    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


BM + Rolling Statistics + Time-Based Features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # # Create lag features with a window size of 24
    # lags = 24
    # for i in range(1, lags + 1):
    #     df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # Add rolling statistics (mean, std, skewness) with a window size of 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    # Add hourly and daily aggregated features
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['weekday'] = df.index.weekday
    df['month'] = df.index.month

    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


MLP BM with all features (rolling statistics,, feature lagging, and time based features)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import math
from bayes_opt import BayesianOptimization  # For Bayesian Search

# Load and preprocess the dataset
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")  # Ensure the correct engine is used for ODS files
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    # Convert 'FECHA_HORA' to datetime and set it as the index
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    # Clean the data: Convert all non-numeric cells to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)  # Drop rows with NaN values
    
    target_col = 'ASOMADILLA-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    # Create lag features with a window size of 24
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    # Add rolling statistics (mean, std, skewness) with a window size of 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    # Add hourly and daily aggregated features
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['weekday'] = df.index.weekday
    df['month'] = df.index.month

    df.dropna(inplace=True)  # Drop rows with NaN values after creating lag and rolling features
    return df, target_col

# Split the data into training and testing sets
def split_data(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    split_index = int(len(df) * 0.8)
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test, scaler

# Build the MLP model
def build_mlp_model(input_dim, num_units=64, dropout_rate=0.2, activation='relu'):
    model = Sequential([
        Dense(num_units, activation=activation, input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(num_units // 2, activation=activation),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Objective function for Bayesian optimization
def objective_function(num_units, dropout_rate, activation, epochs, batch_size):
    activation = ['relu', 'tanh', 'sigmoid'][int(activation)]  # Convert to categorical choice
    num_units = int(num_units)
    epochs = int(epochs)
    batch_size = int(batch_size)
    
    model = build_mlp_model(X_train.shape[1], num_units=num_units, dropout_rate=dropout_rate, activation=activation)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
    val_mae = np.min(history.history['val_mae'])  # Minimize validation MAE
    
    print(f"MAE: {val_mae}, Units: {num_units}, Dropout: {dropout_rate:.3f}, Activation: {activation}, Epochs: {epochs}, Batch Size: {batch_size}")
    return -val_mae  # We return negative because BayesianOptimization maximizes by default

# Hyperparameter optimization function using BayesianOptimization
def optimize_hyperparameters(X_train, y_train):
    pbounds = {
        'num_units': (32, 128),  # Range for number of units
        'dropout_rate': (0.1, 0.5),  # Dropout rate
        'activation': (0, 2),  # 0: relu, 1: tanh, 2: sigmoid
        'epochs': (50, 100),  # Number of epochs
        'batch_size': (16, 64)  # Batch size
    }
    
    optimizer = BayesianOptimization(
        f=lambda num_units, dropout_rate, activation, epochs, batch_size: objective_function(num_units, dropout_rate, activation, epochs, batch_size),
        pbounds=pbounds,
        random_state=42
    )
    
    optimizer.maximize(init_points=5, n_iter=15)  # 5 initial points and 15 iterations of search
    
    print(f"Best Parameters: {optimizer.max['params']}")
    return optimizer.max['params']

# Visualize the training process
def plot_training_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Absolute Error')
    plt.legend()
    plt.title('Training and Validation MAE')
    plt.show()

# Evaluate the model and compute metrics
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = np.mean(np.abs(y_test - y_pred))
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Test MAE: {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    return mae, rmse

# Main function to execute the workflow
def main():
    global X_train, X_test, y_train, y_test  # Needed for the Bayesian objective function
    
    filepath = r"E:\Abroad period research\Time series forecasting\Asomadilla_2015_2023\Asomadilla_2015_2023.ods"
    
    try:
        df, target_col = load_and_preprocess_data(filepath)
        print("Data successfully loaded and cleaned.")
        
        X_train, X_test, y_train, y_test, scaler = split_data(df, target_col)
        
        # Optimize hyperparameters
        best_params = optimize_hyperparameters(X_train, y_train)
        
        # Train the final model with the best parameters
        final_model = build_mlp_model(
            input_dim=X_train.shape[1], 
            num_units=int(best_params['num_units']), 
            dropout_rate=best_params['dropout_rate'], 
            activation=['relu', 'tanh', 'sigmoid'][int(best_params['activation'])]
        )
        
        history = final_model.fit(X_train, y_train, epochs=int(best_params['epochs']), batch_size=int(best_params['batch_size']), validation_split=0.2, verbose=1)
        
        plot_training_history(history)
        mae, rmse = evaluate_model(final_model, X_test, y_test)
    
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()
